In [0]:


query = """
SELECT
    i.order_id,

    COALESCE(c.customer_key,-1) AS customer_key,
    COALESCE(p.product_key,-1) AS product_key,
    COALESCE(s.seller_key,-1) AS seller_key,
    COALESCE(d.date_key,-1) AS date_key,
    COALESCE(g.geo_key,-1) AS geography_key,

    ROUND(i.price,2) AS sales_amount,
    ROUND(i.freight_value,2) AS shipping_cost,
    ROUND(i.price + i.freight_value,2) AS total_order_value

FROM workspace.silver.order_items i

LEFT JOIN workspace.gold.dim_products p
    ON i.product_id = p.product_id

LEFT JOIN workspace.gold.dim_sellers s
    ON i.seller_id = s.seller_id

LEFT JOIN workspace.silver.orders o
    ON i.order_id = o.order_id

LEFT JOIN workspace.gold.dim_customers c
    ON o.customer_order_key = c.customer_order_key

LEFT JOIN workspace.gold.dim_date d
    ON CAST(o.order_purchase_timestamp AS DATE) = d.date

LEFT JOIN workspace.gold.dim_geography g
    ON c.zip_code = g.zip_code
"""



In [0]:
df_fact = spark.sql(query)

(df_fact.write
 .mode("overwrite")
 .format("delta")
 .option("overwriteSchema", "true")
 .saveAsTable("workspace.gold.fact_sales"))

display(spark.table("workspace.gold.fact_sales").limit(10))

It shows that out of 5000 id 4769 are not matching hence and loaded them with a placeholder with ( -1)

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.silver.order_items i
WHERE i.order_id NOT IN (SELECT order_id FROM workspace.silver.orders);